In [2]:
import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as sp_pb2
from transformers import AutoTokenizer
import os
import shutil
from transformers import AutoModelForSeq2SeqLM
import torch
from config import EXTERNAL_DATASET_INPUTS, INTERNAL_DATASET_INPUTS, DICTIONARY_DATASET_INPUTS,Columns
import pandas as pd
from utils import stack_csvs_from_folder

from data_processing.processing import TextProcessor


In [2]:
def save_corpus_to_file(
    akkadian_text: pd.Series, output_path: str = "akkadian_corpus.txt"
) -> str:
    """SentencePiece requires a plain text file as input — one sentence per line."""
    with open(output_path, "w", encoding="utf-8") as f:
        for line in akkadian_text.tolist():
            f.write(line.strip() + "\n")
    print(f"Corpus saved to {output_path}")
    return output_path

In [3]:

def load_akkadian_text() -> pd.Series:
    """Load and concatenate internal and external datasets, return transliteration column."""
    df_internal = stack_csvs_from_folder(INTERNAL_DATASET_INPUTS)
    df_external = stack_csvs_from_folder(EXTERNAL_DATASET_INPUTS)
    df_dictionary = stack_csvs_from_folder(DICTIONARY_DATASET_INPUTS)
    df = pd.concat([df_internal, df_external,df_dictionary], ignore_index=True)

    tp = TextProcessor()

    df[Columns.TRANSLITERATION] = df[Columns.TRANSLITERATION].apply(
        lambda x: tp.preprocess_transliteration_text(
            transliteration_text=x,
            separate_compounds=False,
            with_hyphens=False,
            named_determinatives=False,
            normalize_chars=False,
            diacritic_mode=True
        )
    )
    return df[Columns.TRANSLITERATION].dropna()


In [37]:

def train_akkadian_tokenizer(corpus_path, vocab_size=2500):

    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix="akkadian_sp",
        vocab_size=vocab_size,
        model_type="bpe",
        user_defined_symbols=["<lgg_s>", "<lgg_e>"],
        character_coverage=0.98,
        split_by_whitespace=True,
        split_by_unicode_script=True,
        byte_fallback=True,
        max_sentencepiece_length=16,
        input_sentence_size=50000,
        shuffle_input_sentence=True,
        normalization_rule_name="identity"
    )

    print("Akkadian tokenizer trained.")

In [5]:


def merge_sentencepiece_models(
    mbart_sp_model,
    akkadian_sp_model,
    output_model="merged_mbart_akkadian.model"
):

    mbart = sp_pb2.ModelProto()
    akk = sp_pb2.ModelProto()

    mbart.ParseFromString(open(mbart_sp_model, "rb").read())
    akk.ParseFromString(open(akkadian_sp_model, "rb").read())

    mbart_tokens = {p.piece for p in mbart.pieces}

    added = 0

    for piece in akk.pieces:

        if piece.piece not in mbart_tokens:

            new_piece = mbart.pieces.add()
            new_piece.piece = piece.piece
            new_piece.score = -10.0
            new_piece.type = piece.type

            added += 1

    with open(output_model, "wb") as f:
        f.write(mbart.SerializeToString())

    print("Merged tokenizer created.")
    print("New tokens added:", added)

    return output_model

In [39]:

def load_extended_tokenizer(base_model, merged_sp_model, save_dir):

    tokenizer = AutoTokenizer.from_pretrained(base_model)

    os.makedirs(save_dir, exist_ok=True)

    tokenizer.save_pretrained(save_dir)

    spm_path = os.path.join(save_dir, "sentencepiece.bpe.model")

    shutil.copy(merged_sp_model, spm_path)

    tokenizer = AutoTokenizer.from_pretrained(save_dir)

    print("Tokenizer loaded with merged SentencePiece.")

    return tokenizer

In [6]:

def load_extended_model(base_model, tokenizer):

    model = AutoModelForSeq2SeqLM.from_pretrained(base_model)

    old_vocab = model.get_input_embeddings().weight.shape[0]

    model.resize_token_embeddings(len(tokenizer))

    with torch.no_grad():

        emb = model.get_input_embeddings().weight
        mean_emb = emb[:old_vocab].mean(dim=0)

        emb[old_vocab:] = mean_emb

    print("Embedding resized:", old_vocab, "->", len(tokenizer))

    return model

In [4]:
import sentencepiece as spm


def debug_tokenizer(tokenizer, akkadian_sp_model, sample_text):

    sp = spm.SentencePieceProcessor(model_file=akkadian_sp_model)

    akk_tokens = {sp.id_to_piece(i) for i in range(sp.get_piece_size())}

    tokens = tokenizer.tokenize(sample_text)

    used = [t for t in tokens if t in akk_tokens]

    print("\nDEBUG TOKENIZATION")
    print("------------------")
    print("Input text:", sample_text)
    print("Tokens:", tokens)
    print("Akkadian tokens used:", used)
    print("Count:", len(used))

In [25]:
BASE_MODEL = "MarkSpaghetti/mbart-lora-r32-alpha-16"


akkadian_text = load_akkadian_text()
save_corpus_to_file(akkadian_text)
CORPUS = "akkadian_corpus.txt"

train_akkadian_tokenizer(CORPUS)


# tokenizer_tmp = AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=False)
# mbart_sp_model = tokenizer_tmp.vocab_file


# merged_model = merge_sentencepiece_models(
#     mbart_sp_model,
#     "akkadian_sp.model"
# )


# tokenizer = load_extended_tokenizer(
#     BASE_MODEL,
#     merged_model,
#     "mbart-akkadian"
# )


# model = load_extended_model(BASE_MODEL, tokenizer)



Corpus saved to akkadian_corpus.txt
Akkadian tokenizer trained.


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: akkadian_corpus.txt
  input_format: 
  model_prefix: akkadian_sp
  model_type: BPE
  vocab_size: 2500
  self_test_sample_size: 0
  character_coverage: 0.98
  input_sentence_size: 50000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0

In [26]:
debug_tokenizer(
    tokenizer,
    "akkadian_sp.model",
    ""
)

NameError: name 'debug_tokenizer' is not defined

In [16]:
sp = spm.SentencePieceProcessor(model_file="akkadian_vocab_sp.model")

new_tokens = []

for i in range(sp.get_piece_size()):
    piece = sp.id_to_piece(i)

    if piece not in ["<unk>", "<s>", "</s>"]:
        new_tokens.append(piece)

print("Tokens extracted:", len(new_tokens))



Tokens extracted: 1497


In [17]:
print("<gap>" in new_tokens)
new_tokens.remove("<gap>")
print("<gap>" in new_tokens)


True
False


In [7]:
akk_tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")
model = AutoModelForSeq2SeqLM.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")
eng_tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")

Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [18]:
eng_tokenizer.add_special_tokens({
    'additional_special_tokens': ['<gap>']
})

akk_tokenizer.add_special_tokens({
    'additional_special_tokens': ['<gap>']
})

0

In [19]:
print(len(eng_tokenizer)) 
print()
print(len(akk_tokenizer)) 

assert eng_tokenizer.convert_tokens_to_ids("<gap>") == akk_tokenizer.convert_tokens_to_ids("<gap>")

250056

250056


In [20]:
print(len(new_tokens))

1496


In [21]:
# CREATE THE AKKADIAN TOKENIZER
print(len(akk_tokenizer))
num_added_toks = akk_tokenizer.add_tokens(new_tokens)
print("We have added", num_added_toks, "tokens")
print(len(akk_tokenizer))

250056
We have added 1496 tokens
250899


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

#model         = AutoModelForSeq2SeqLM.from_pretrained("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16")
eng_tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16", subfolder="english_tokenizer")
akk_tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16", subfolder="akkadian_tokenizer")

In [8]:
import time

# Time just the tokenization
start = time.time()
for i in range(100):
    akk_tokenizer("a-na dumu šu", max_length=512, padding="max_length", truncation=True)
print("Akkadian tokenizer:", time.time() - start)

start = time.time()
for i in range(100):
    eng_tokenizer("the king went to the city", max_length=512, padding="max_length", truncation=True)
print("English tokenizer:", time.time() - start)

Akkadian tokenizer: 0.017427444458007812
English tokenizer: 0.015843629837036133


In [ ]:
eng_tokenizer

In [5]:
# tokenizer.src_lang = "en_XX"  # or whatever your source language is
tp = TextProcessor()
akk_text = "IGI ša-lim-a-šur DUMU i-lí-a IGI sá-sí-a" 
akk_text = tp.preprocess_transliteration_text(
            transliteration_text=akk_text,
            separate_compounds=False,
            with_hyphens=False,
            named_determinatives=False,
            normalize_chars=False,
            diacritic_mode=True
        )
test_text = akk_text + " Observed by Šalim-Aššur son of Iliya, by Sasiya. He went to the shop to eat vegetables"
print("AKKADIAN")
ids = akk_tokenizer.tokenize(test_text)
print(ids)

print("ENGLISH")

ids = eng_tokenizer.tokenize(test_text)
print(ids)

AKKADIAN
['<lgg_s>', 'igi', '<lgg_e>', '▁', 'šalim', 'ašur', '▁', '<lgg_s>', 'dumu', '<lgg_e>', '▁', 'il', 'í', 'a', '▁', '<lgg_s>', 'igi', '<lgg_e>', '▁', 'sá', 'sí', 'a', '▁O', 'b', 'se', 'r', 'v', 'e', 'd', '▁', 'b', 'y', '▁Š', 'ali', 'm', '-', '▁A', 'š', 'šur', '▁', 's', 'o', 'n', '▁', 'o', 'f', '▁I', 'li', 'y', 'a', '▁', ',', '▁', 'b', 'y', '▁S', 'a', 'si', 'y', 'a', '.', '▁H', 'e', '▁', 'w', 'en', 't', '▁', 't', 'o', '▁', 't', 'h', 'e', '▁', 's', 'h', 'o', 'p', '▁', 't', 'o', '▁', 'e', 'at', '▁', 'v', 'e', 'g', 'et', 'ab', 'l', 'e', 's']
ENGLISH
['▁<', 'l', 'gg', '_', 's', '>', 'igi', '<', 'l', 'gg', '_', 'e', '>', '▁', 'šali', 'maš', 'ur', '▁<', 'l', 'gg', '_', 's', '>', 'dumu', '<', 'l', 'gg', '_', 'e', '>', '▁il', 'ía', '▁<', 'l', 'gg', '_', 's', '>', 'igi', '<', 'l', 'gg', '_', 'e', '>', '▁sá', 's', 'ía', '▁Ob', 'serve', 'd', '▁by', '▁Šal', 'im', '-', 'A', 'š', 'šu', 'r', '▁son', '▁of', '▁Il', 'iya', ',', '▁by', '▁S', 'asiya', '.', '▁He', '▁went', '▁to', '▁the', '▁shop', '▁to

In [24]:
old_vocab_size = model.model.shared.weight.shape[0]
model.resize_token_embeddings(len(akk_tokenizer))

with torch.no_grad():
    mean_embedding = model.model.shared.weight[:old_vocab_size].mean(dim=0)
    model.model.shared.weight[old_vocab_size:] = mean_embedding

print(f"Embedding matrix resized: {old_vocab_size} -> {len(akk_tokenizer)}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding matrix resized: 250055 -> 250899


In [28]:
model.push_to_hub("MarkSpaghetti/mbart-tokens-lora-r32-alpha-16")

eng_tokenizer.push_to_hub(
    "MarkSpaghetti/mbart-tokens-lora-r32-alpha-16",
    subfolder="english_tokenizer"
)

akk_tokenizer.push_to_hub(
    "MarkSpaghetti/mbart-tokens-lora-r32-alpha-16",
    subfolder="akkadian_tokenizer"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [27]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "MarkSpaghetti/mbart-tokens-lora-r32-alpha-16"

# Push model normally
model.push_to_hub(repo_id)

# Save tokenizers locally first
eng_tokenizer.save_pretrained("./eng_tokenizer")
akk_tokenizer.save_pretrained("./akk_tokenizer")

# Push each file manually into the correct subfolder
api.upload_folder(
    folder_path="./eng_tokenizer",
    repo_id=repo_id,
    path_in_repo="english_tokenizer"  # this becomes the subfolder on the hub
)

api.upload_folder(
    folder_path="./akk_tokenizer",
    repo_id=repo_id,
    path_in_repo="akkadian_tokenizer"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/MarkSpaghetti/mbart-tokens-lora-r32-alpha-16/commit/68426caa44610897fdab7beff112e0c57f0e25f6', commit_message='Upload folder using huggingface_hub', commit_description='', oid='68426caa44610897fdab7beff112e0c57f0e25f6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MarkSpaghetti/mbart-tokens-lora-r32-alpha-16', endpoint='https://huggingface.co', repo_type='model', repo_id='MarkSpaghetti/mbart-tokens-lora-r32-alpha-16'), pr_revision=None, pr_num=None)